# ConvNeXt Object Detection — Faster R-CNN (PyTorch)

**Backbone:** ConvNeXt-Tiny (pretrained on ImageNet)  
**Detection Head:** Faster R-CNN with FPN  
**Dataset:** PASCAL VOC 2012 (20 object classes + background)

**Classes:** aeroplane, bicycle, bird, boat, bottle, bus, car, cat, chair, cow, dining table, dog, horse, motorbike, person, potted plant, sheep, sofa, train, tv/monitor

**Architecture:**
```
ConvNeXt-Tiny (backbone) → FPN (Feature Pyramid Network) → RPN → ROI Heads → Boxes + Labels + Scores
```

> GPU accelerator ဖွင့်ပြီး run ပါ: `Settings → Accelerator → GPU`

In [ ]:
# --- 1. Install Dependencies ---
# !pip install -q torch torchvision pycocotools matplotlib tqdm

In [ ]:
# --- 2. Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms, models
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.ops import misc as misc_nn_ops
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os
import random
import time
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("✅ All imports successful.")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")

In [ ]:
# --- 3. Device & Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Hyperparameters
IMG_SIZE = 512
BATCH_SIZE = 4          # Object detection needs more memory
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
FINE_TUNE_LR = 1e-5
NUM_WORKERS = 2
FREEZE_EPOCHS = 3

# VOC classes (background = 0, so 21 total)
VOC_CLASSES = [
    '__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog',
    'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa',
    'train', 'tvmonitor'
]
NUM_CLASSES = len(VOC_CLASSES)  # 21
print(f"Classes ({NUM_CLASSES}): {VOC_CLASSES[1:]}")  # skip background

## 📦 Dataset — PASCAL VOC 2012

VOC dataset ကို `torchvision.datasets.VOCDetection` နဲ့ download လုပ်ပါမယ်။  
Annotations (XML) ကနေ bounding boxes + labels ကို parse လုပ်ပါမယ်။

In [ ]:
# --- 4. VOC Detection Dataset ---
class VOCDetectionDataset(Dataset):
    """PASCAL VOC format dataset for object detection."""
    def __init__(self, root, year='2012', image_set='train', transforms=None):
        self.voc = torchvision.datasets.VOCDetection(
            root=root, year=year, image_set=image_set, download=True
        )
        self.transforms = transforms
        self.class_to_idx = {cls: i for i, cls in enumerate(VOC_CLASSES)}

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        img, annotation = self.voc[idx]
        img = img.convert("RGB")

        # Parse VOC XML annotation
        objects = annotation['annotation']['object']
        if not isinstance(objects, list):
            objects = [objects]

        boxes = []
        labels = []
        for obj in objects:
            cls_name = obj['name']
            if cls_name not in self.class_to_idx:
                continue
            bbox = obj['bndbox']
            xmin = float(bbox['xmin'])
            ymin = float(bbox['ymin'])
            xmax = float(bbox['xmax'])
            ymax = float(bbox['ymax'])

            # Skip invalid boxes
            if xmax <= xmin or ymax <= ymin:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[cls_name])

        # Handle images with no valid objects
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.tensor([]),
            'iscrowd': torch.zeros(len(boxes), dtype=torch.int64)
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

print("✅ VOCDetectionDataset defined.")

In [ ]:
# --- 5. Transforms & DataLoaders ---
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Download & Load VOC 2012
print("⏳ Downloading PASCAL VOC 2012...")
train_dataset = VOCDetectionDataset('./data', year='2012', image_set='train', transforms=train_transform)
val_dataset = VOCDetectionDataset('./data', year='2012', image_set='val', transforms=val_transform)

def collate_fn(batch):
    """Custom collate — detection targets have variable number of boxes."""
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)

print(f"✅ Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print(f"   Batches → Train: {len(train_loader)} | Val: {len(val_loader)}")

In [ ]:
# --- 6. Visualize VOC Samples with Bounding Boxes ---
COLORS = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))

def unnormalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    img = img.clone()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(img, 0, 1)

def show_detection(img_tensor, target, class_names=VOC_CLASSES, ax=None):
    """Display image with bounding boxes."""
    if ax is None:
        fig, ax = plt.subplots(1, figsize=(8, 8))
    img = unnormalize(img_tensor).permute(1, 2, 0).numpy()
    ax.imshow(img)

    boxes = target['boxes']
    labels = target['labels']
    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box.numpy()
        # Scale boxes to display size
        w, h = x2 - x1, y2 - y1
        color = COLORS[label % len(COLORS)]
        rect = patches.Rectangle((x1, y1), w, h, linewidth=2,
                                  edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, class_names[label],
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor=color, alpha=0.7, pad=1))
    ax.axis('off')

# Show 6 samples
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, ax in enumerate(axes.flatten()):
    idx = random.randint(0, len(train_dataset) - 1)
    img, target = train_dataset[idx]
    show_detection(img, target, ax=ax)
    ax.set_title(f"Image #{idx} — {len(target['boxes'])} objects", fontsize=10)

plt.suptitle("PASCAL VOC 2012 — Sample Training Images with Bounding Boxes",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🏗️ ConvNeXt + Faster R-CNN Model

ConvNeXt-Tiny backbone ကို Feature Pyramid Network (FPN) နဲ့ ချိတ်ဆက်ပြီး Faster R-CNN detection head ထည့်ပါမယ်။

```
ConvNeXt-Tiny Backbone
    ↓ (multi-scale feature maps)
FPN (Feature Pyramid Network)
    ↓
RPN (Region Proposal Network) → Proposals
    ↓
ROI Align → Box Regression + Classification
    ↓
Boxes + Labels + Confidence Scores
```

In [ ]:
# --- 7. Build ConvNeXt + Faster R-CNN ---
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def build_convnext_fasterrcnn(num_classes, pretrained_backbone=True):
    """
    Build Faster R-CNN with ConvNeXt-Tiny backbone + FPN.
    """
    # Load pretrained ConvNeXt-Tiny
    weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained_backbone else None
    convnext = models.convnext_tiny(weights=weights)

    # ConvNeXt feature stages & output channels
    # Stage 0: features.0-1 → 96ch  (H/4)
    # Stage 1: features.2-3 → 192ch (H/8)
    # Stage 2: features.4-5 → 384ch (H/16)
    # Stage 3: features.6-7 → 768ch (H/32)

    # Extract backbone feature layers
    backbone_features = convnext.features

    class ConvNeXtBackbone(nn.Module):
        def __init__(self, features):
            super().__init__()
            # Split into 4 stages
            self.stage1 = nn.Sequential(features[0], features[1])  # stem + stage1
            self.stage2 = nn.Sequential(features[2], features[3])  # downsample + stage2
            self.stage3 = nn.Sequential(features[4], features[5])  # downsample + stage3
            self.stage4 = nn.Sequential(features[6], features[7])  # downsample + stage4
            self.out_channels = 256  # FPN output

        def forward(self, x):
            c1 = self.stage1(x)  # (B, 96, H/4, W/4)
            c2 = self.stage2(c1) # (B, 192, H/8, W/8)
            c3 = self.stage3(c2) # (B, 384, H/16, W/16)
            c4 = self.stage4(c3) # (B, 768, H/32, W/32)
            return {'0': c1, '1': c2, '2': c3, '3': c4}

    backbone = ConvNeXtBackbone(backbone_features)

    # Build FPN on top
    in_channels_list = [96, 192, 384, 768]
    fpn = torchvision.ops.FeaturePyramidNetwork(in_channels_list, out_channels=256)

    class BackboneFPN(nn.Module):
        def __init__(self, body, fpn):
            super().__init__()
            self.body = body
            self.fpn = fpn
            self.out_channels = 256

        def forward(self, x):
            features = self.body(x)
            return self.fpn(features)

    backbone_fpn = BackboneFPN(backbone, fpn)

    # Anchor generator for multi-scale features
    anchor_sizes = ((32,), (64,), (128,), (256,), (512,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

    # ROI pooler
    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', '3'],
        output_size=7,
        sampling_ratio=2
    )

    # Build Faster R-CNN
    model = FasterRCNN(
        backbone=backbone_fpn,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
        min_size=IMG_SIZE,
        max_size=IMG_SIZE
    )

    return model

model = build_convnext_fasterrcnn(NUM_CLASSES, pretrained_backbone=True).to(device)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters:     {total_params:>12,}")
print(f"Trainable Parameters: {trainable_params:>12,}")
print(f"✅ ConvNeXt + Faster R-CNN built successfully.")

In [ ]:
# --- 8. Optimizer & LR Scheduler ---
# Phase 1: freeze backbone, train only FPN + detection heads
for name, param in model.named_parameters():
    if 'backbone.body' in name:
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 — Trainable params (head only): {trainable_params:,}")

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("✅ Optimizer & Scheduler configured.")

## 🚀 Training Loop

Faster R-CNN training:
- Model ကို **train mode** ထားရင် `model(images, targets)` က **loss dict** return ပြန်ပေးမယ်
- `eval mode` ဆို **predictions** (boxes, labels, scores) return ပြန်ပေးမယ်

**2-Phase Training:**
1. Phase 1: Backbone frozen → FPN + detection heads only
2. Phase 2: Unfreeze backbone → full fine-tune with low LR

In [ ]:
# --- 9. Training Loop ---
train_losses = []
best_loss = float('inf')
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # === Phase 2: Unfreeze backbone ===
    if epoch == FREEZE_EPOCHS:
        print(f"\n{'='*60}")
        print("🔓 Phase 2: Unfreezing ConvNeXt backbone for fine-tuning!")
        print(f"{'='*60}")
        for param in model.parameters():
            param.requires_grad = True

        optimizer = optim.AdamW([
            {'params': model.backbone.body.parameters(), 'lr': FINE_TUNE_LR},
            {'params': model.backbone.fpn.parameters(), 'lr': LEARNING_RATE},
            {'params': model.rpn.parameters(), 'lr': LEARNING_RATE},
            {'params': model.roi_heads.parameters(), 'lr': LEARNING_RATE},
        ], weight_decay=1e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   Trainable parameters: {trainable:,}\n")

    # === Train ===
    model.train()
    epoch_loss = 0.0
    phase = "Phase 1 (Frozen)" if epoch < FREEZE_EPOCHS else "Phase 2 (Fine-tune)"

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward — Faster R-CNN returns losses in train mode
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += losses.item()
        pbar.set_postfix({
            'loss': f"{losses.item():.4f}",
            'cls': f"{loss_dict.get('loss_classifier', 0):.3f}",
            'box': f"{loss_dict.get('loss_box_reg', 0):.3f}",
        })

    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"[{phase}] Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Avg Loss: {avg_loss:.4f} | LR: {current_lr:.2e}")

    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "best_convnext_fasterrcnn_voc.pth")
        print(f"  → 💾 Best model saved! (Loss: {avg_loss:.4f})")

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"✅ Training Complete! Time: {elapsed/60:.1f} min")
print(f"   Best Loss: {best_loss:.4f}")
print(f"{'='*60}")

In [ ]:
# --- 10. Plot Training Loss ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS+1), train_losses, 'b-o', markersize=5, label='Training Loss')
plt.axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle='--', alpha=0.7, label='Backbone Unfreeze')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('ConvNeXt + Faster R-CNN — Training Loss', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("convnext_detection_loss.png", dpi=150, bbox_inches='tight')
plt.show()

## 🧪 Evaluation & Visualization

Best model ကို load ပြန်လုပ်ပြီး validation set ပေါ်မှာ prediction results ကြည့်ပါမယ်။

In [ ]:
# --- 11. Load Best Model & Run Predictions ---
model.load_state_dict(torch.load("best_convnext_fasterrcnn_voc.pth",
                                  map_location=device, weights_only=True))
model.eval()

def predict_and_show(model, dataset, num_images=6, score_threshold=0.5):
    """Run prediction and visualize results with confidence scores."""
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    axes = axes.flatten()
    indices = random.sample(range(len(dataset)), num_images)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            img, gt_target = dataset[idx]
            predictions = model([img.to(device)])[0]

            # Filter by confidence threshold
            keep = predictions['scores'] >= score_threshold
            boxes = predictions['boxes'][keep].cpu()
            labels = predictions['labels'][keep].cpu()
            scores = predictions['scores'][keep].cpu()

            # Display
            img_display = unnormalize(img).permute(1, 2, 0).numpy()
            axes[i].imshow(img_display)

            for box, label, score in zip(boxes, labels, scores):
                x1, y1, x2, y2 = box.numpy()
                color = COLORS[label.item() % len(COLORS)]
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                          edgecolor=color, facecolor='none')
                axes[i].add_patch(rect)
                axes[i].text(x1, y1 - 5,
                             f"{VOC_CLASSES[label.item()]} {score:.2f}",
                             color='white', fontsize=8, fontweight='bold',
                             bbox=dict(facecolor=color, alpha=0.8, pad=1))

            axes[i].set_title(f"Detected: {len(boxes)} objects (threshold: {score_threshold})",
                              fontsize=10)
            axes[i].axis('off')

    plt.suptitle("ConvNeXt + Faster R-CNN — Detection Results on VOC 2012",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("convnext_detection_results.png", dpi=150, bbox_inches='tight')
    plt.show()

predict_and_show(model, val_dataset, num_images=6, score_threshold=0.5)

In [ ]:
# --- 12. mAP Evaluation (Mean Average Precision) ---
from collections import defaultdict

def compute_iou(box1, box2):
    """Compute IoU between two sets of boxes."""
    x1 = torch.max(box1[:, 0].unsqueeze(1), box2[:, 0].unsqueeze(0))
    y1 = torch.max(box1[:, 1].unsqueeze(1), box2[:, 1].unsqueeze(0))
    x2 = torch.min(box1[:, 2].unsqueeze(1), box2[:, 2].unsqueeze(0))
    y2 = torch.min(box1[:, 3].unsqueeze(1), box2[:, 3].unsqueeze(0))
    inter = torch.clamp(x2 - x1, min=0) * torch.clamp(y2 - y1, min=0)
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    union = area1.unsqueeze(1) + area2.unsqueeze(0) - inter
    return inter / (union + 1e-6)

def evaluate_map(model, dataloader, iou_threshold=0.5, score_threshold=0.3, max_batches=50):
    """Compute per-class AP and mAP@0.5."""
    model.eval()
    all_detections = defaultdict(list)  # class_id -> [(score, is_tp)]
    total_gt = defaultdict(int)

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(tqdm(dataloader, desc="Evaluating")):
            if batch_idx >= max_batches:
                break
            images = [img.to(device) for img in images]
            predictions = model(images)

            for pred, gt in zip(predictions, targets):
                gt_boxes = gt['boxes']
                gt_labels = gt['labels']
                pred_boxes = pred['boxes'].cpu()
                pred_labels = pred['labels'].cpu()
                pred_scores = pred['scores'].cpu()

                # Count ground truths
                for lbl in gt_labels:
                    total_gt[lbl.item()] += 1

                # Filter predictions
                keep = pred_scores >= score_threshold
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]
                pred_scores = pred_scores[keep]

                matched_gt = set()
                for score, p_box, p_lbl in sorted(
                    zip(pred_scores, pred_boxes, pred_labels),
                    key=lambda x: x[0], reverse=True
                ):
                    cls = p_lbl.item()
                    if len(gt_boxes) > 0:
                        ious = compute_iou(p_box.unsqueeze(0), gt_boxes).squeeze(0)
                        same_cls = (gt_labels == cls)
                        ious[~same_cls] = 0
                        max_iou, max_idx = ious.max(0) if len(ious) > 0 else (torch.tensor(0), torch.tensor(0))
                        if max_iou >= iou_threshold and max_idx.item() not in matched_gt:
                            all_detections[cls].append((score.item(), True))
                            matched_gt.add(max_idx.item())
                        else:
                            all_detections[cls].append((score.item(), False))
                    else:
                        all_detections[cls].append((score.item(), False))

    # Compute AP per class
    aps = {}
    for cls_id in range(1, NUM_CLASSES):  # skip background
        dets = sorted(all_detections.get(cls_id, []), key=lambda x: x[0], reverse=True)
        tp = np.array([d[1] for d in dets], dtype=np.float32)
        total = total_gt.get(cls_id, 0)
        if total == 0:
            continue
        tp_cumsum = np.cumsum(tp)
        fp_cumsum = np.cumsum(1 - tp)
        recall = tp_cumsum / total
        precision = tp_cumsum / (tp_cumsum + fp_cumsum)
        # AP using 11-point interpolation
        ap = 0
        for t in np.arange(0, 1.1, 0.1):
            p = precision[recall >= t].max() if np.any(recall >= t) else 0
            ap += p / 11
        aps[VOC_CLASSES[cls_id]] = ap

    mAP = np.mean(list(aps.values())) if aps else 0
    return aps, mAP

aps, mAP = evaluate_map(model, val_loader, iou_threshold=0.5, max_batches=50)

print(f"\n{'='*50}")
print(f"{'Class':<20} {'AP@0.5':>10}")
print(f"{'='*50}")
for cls, ap in sorted(aps.items(), key=lambda x: x[1], reverse=True):
    print(f"{cls:<20} {ap:>10.4f}")
print(f"{'='*50}")
print(f"{'mAP@0.5':<20} {mAP:>10.4f}")
print(f"{'='*50}")

In [ ]:
# --- 13. Per-Class AP Visualization ---
if aps:
    sorted_aps = dict(sorted(aps.items(), key=lambda x: x[1], reverse=True))
    plt.figure(figsize=(12, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_aps)))
    bars = plt.barh(list(sorted_aps.keys())[::-1], list(sorted_aps.values())[::-1],
                    color=colors[::-1], edgecolor='white')
    plt.xlabel('Average Precision (AP@0.5)', fontsize=12)
    plt.title(f'Per-Class AP — ConvNeXt + Faster R-CNN (mAP@0.5: {mAP:.4f})',
              fontsize=13, fontweight='bold')
    plt.xlim(0, 1)
    for bar, val in zip(bars, list(sorted_aps.values())[::-1]):
        plt.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig("convnext_detection_ap.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- 14. Single Image Inference ---
def detect_objects(model, image_path, score_threshold=0.5):
    """Run object detection on a single image."""
    model.eval()
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transform(img_pil).to(device)

    with torch.no_grad():
        predictions = model([img_tensor])[0]

    keep = predictions['scores'] >= score_threshold
    boxes = predictions['boxes'][keep].cpu()
    labels = predictions['labels'][keep].cpu()
    scores = predictions['scores'][keep].cpu()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # (A) Detection result
    img_display = unnormalize(img_tensor.cpu()).permute(1, 2, 0).numpy()
    axes[0].imshow(img_display)
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box.numpy()
        color = COLORS[label.item() % len(COLORS)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                  edgecolor=color, facecolor='none')
        axes[0].add_patch(rect)
        axes[0].text(x1, y1-5, f"{VOC_CLASSES[label.item()]} {score:.2f}",
                     color='white', fontsize=9, fontweight='bold',
                     bbox=dict(facecolor=color, alpha=0.8, pad=1))
    axes[0].set_title(f"Detections ({len(boxes)} objects)", fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # (B) Confidence scores
    if len(scores) > 0:
        det_names = [f"{VOC_CLASSES[l.item()]} #{i+1}" for i, l in enumerate(labels)]
        colors_bar = [COLORS[l.item() % len(COLORS)] for l in labels]
        axes[1].barh(det_names[::-1], scores.numpy()[::-1], color=colors_bar[::-1])
        axes[1].set_xlabel('Confidence Score')
        axes[1].set_title('Detection Confidence', fontsize=12, fontweight='bold')
        axes[1].set_xlim(0, 1)
    else:
        axes[1].text(0.5, 0.5, 'No detections', ha='center', va='center', fontsize=14)
        axes[1].axis('off')

    plt.suptitle("ConvNeXt + Faster R-CNN — Single Image Detection", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return boxes, labels, scores

# Usage:
# detect_objects(model, "path/to/image.jpg")
print("✅ detect_objects() ready. Usage:")
print('   detect_objects(model, "path/to/image.jpg", score_threshold=0.5)')